In [ ]:
from pyspark.sql.utils import AnalysisException

# ---------------- CONFIGURACIÓN ----------------
control_table_name = "dbo.bronze_to_silver_control" 
default_source_system = "QAD"
new_column_name = "source_system"

# --- CONFIGURACIÓN DE POSICIÓN ---
# Opciones: "FIRST", "AFTER", "END"
# "FIRST": Al inicio de la tabla.
# "AFTER": Después de una columna especifica.
# "END": Al final (comportamiento por defecto).
position_mode = "AFTER" 

# (Si esta columna no existe en la tabla destino, se agregará al final).
reference_column = "company_code" 

# ---------------- PASO 1: LEER TABLA DE CONTROL ----------------
print(f"Leyendo tabla de control: {control_table_name}...")

try:
    df_control = spark.sql(f"""
        SELECT target_layer, target_schema, target_table 
        FROM {control_table_name} 
        WHERE is_active = 1 
    """)
    tables_to_process = df_control.collect()
    print(f"Se encontraron {len(tables_to_process)} tablas activas.")

except AnalysisException as e:
    print(f"Error leyendo control: {e}")
    tables_to_process = []

# ---------------- PASO 2: ITERAR Y MODIFICAR ----------------

for row in tables_to_process:
    full_table_name = f"{row['target_layer']}.{row['target_schema']}.{row['target_table']}"
    print(f"--- Procesando: {full_table_name} ---")
    
    try:
        if spark.catalog.tableExists(full_table_name):
            target_df = spark.table(full_table_name)
            current_columns = target_df.columns
            
            if new_column_name in current_columns:
                print(f"⚠️ La columna '{new_column_name}' ya existe. Saltando...")
            else:
                # --- Lógica de Posicionamiento ---
                position_clause = ""
                
                if position_mode == "FIRST":
                    position_clause = "FIRST"
                    print("Posición: Al inicio de la tabla.")
                
                elif position_mode == "AFTER":
                    if reference_column in current_columns:
                        position_clause = f"AFTER {reference_column}"
                        print(f"Posición: Después de '{reference_column}'.")
                    else:
                        print(f"⚠️ La columna de referencia '{reference_column}' no existe en esta tabla. Se agregará al final.")
                        position_clause = "" # Se agrega al final
                
                # --- Ejecución ALTER TABLE ---
                # Sintaxis: ALTER TABLE tbl ADD COLUMNS (col type [FIRST | AFTER col])
                alter_query = f"""
                    ALTER TABLE {full_table_name} 
                    ADD COLUMNS ({new_column_name} STRING {position_clause})
                """
                
                spark.sql(alter_query)
                
                # --- Ejecución UPDATE (Backfill) ---
                print(f"Llenando datos con '{default_source_system}'...")
                spark.sql(f"""
                    UPDATE {full_table_name} 
                    SET {new_column_name} = '{default_source_system}'
                    WHERE {new_column_name} IS NULL
                """)
                
                print(f"✅ Éxito.")
        else:
            print(f"❌ Error: Tabla no encontrada.")

    except Exception as e:
        print(f"❌ Error crítico en {full_table_name}: {str(e)}")

print("\n--- Proceso Finalizado ---")
